## Fourier coefficients

This notebook file demonstrates the calculations of Fourier coefficients.

- Create a triangle function.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------
# Triangle wave Fourier series: up to 6 odd harmonics
# ----------------------------------

# Domain (one period = 2π). Change to e.g. [-3*np.pi, 3*np.pi] to show more periods.
x = np.linspace(0.0, 2.0 * np.pi, 4000)

# Exact triangle wave with amplitude ±1 and period 2π
tri_exact = (2.0 / np.pi) * np.arcsin(np.sin(0.5*x))

# Plot once just to mimic the first simple plot (optional)
plt.figure()
plt.plot(x, tri_exact)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Exact triangle wave")
plt.grid(True)
plt.tight_layout()


A function can be rewritten in a Fourier series as

$$f(x) = a_0 + \sum_{n=0}^\infty \bigg( a_n \cos (nx) + b_n \sin (nx) \bigg),$$

where 
$$a_0 = \frac{1}{2 \pi} \int_{-\pi}^{\pi} f(x) dx$$

$$a_n = \frac{1}{\pi} \int_{-\pi}^{\pi} f(x) \cos (nx) dx ~~\text{for}~~ n = 1, 2, ..., \infty$$

$$b_n = \frac{1}{\pi} \int_{-\pi}^{\pi} f(x) \sin (nx) dx ~~\text{for}~~ n = 1, 2, ..., \infty.$$

- Calculate Fourier coefficients


In [ ]:
import numpy as np

def fourier_coeffs_2pi(f, N, M=10000):
    """
    Compute Fourier series coefficients a0, an, bn for a 2π-periodic function f(x).
    
    f : callable, f(x) with x in [0, 2π]
    N : highest harmonic n
    M : number of sample points in [0, 2π)
    """
    # sample points
    x = np.linspace(0.0, 2.0*np.pi, M, endpoint=False)
    dx = x[1] - x[0]
    fx = f(x)
    
    # a0
    a0 = (1.0/np.pi) * np.sum(fx) * dx
    
    # an, bn
    n = np.arange(1, N+1)[:, None]     # shape (N,1) for broadcasting
    cos_nx = np.cos(n * x[None, :])    # shape (N,M)
    sin_nx = np.sin(n * x[None, :])    # shape (N,M)
    
    an = (1.0/np.pi) * np.sum(fx * cos_nx, axis=1) * dx  # shape (N,)
    bn = (1.0/np.pi) * np.sum(fx * sin_nx, axis=1) * dx  # shape (N,)
    
    return a0, an, bn

- Plot the harmonics

In [ ]:
# ----------------------------------
# Figure 1: individual harmonics (for N = Nmax)
# ----------------------------------
import numpy as np
import matplotlib.pyplot as plt

# --- define the function you want to expand ---
def tri_exact(x):
    return (2/np.pi) * np.arcsin(np.sin(0.5*x))

# assume: a0, an, bn already computed, and t defined
# t = np.linspace(0, 2*np.pi, 2000)

plt.figure(figsize=(8,5))
plt.box(True)
plt.grid(True)

Nmax = 6
N = Nmax
a0, an, bn = fourier_coeffs_2pi(tri_exact, N)

print("a0 = ", a0)
print("an = ", an)
print("bn = ", bn)

N = len(an)
colors = plt.cm.tab10(np.linspace(0, 1, N))

for n in range(1, N+1):
    # single harmonic contribution:
    # f_n(x) = a_n cos(nx) + b_n sin(nx)
    fn = an[n-1] * np.cos(n * t) + bn[n-1] * np.sin(n * t)

    plt.plot(t,fn,color=colors[n-1],linewidth=2,label=f"n = {n}")

plt.xlabel("x")
plt.ylabel("harmonic amplitude")
plt.title("Individual Fourier harmonics")
plt.legend()
plt.show()

- Reconstruct a function based on Fourier series.

In [ ]:
def reconstruct_from_coeffs(x, a0, an, bn):
    fN = a0 / 2.0 * np.ones_like(x)
    N = len(an)
    for n in range(1, N+1):
        fN += an[n-1] * np.cos(n * x) + bn[n-1] * np.sin(n * x)
    return fN

- Add all harmonics to reconstruct the function.

In [ ]:
import matplotlib.pyplot as plt

# --- define the function you want to expand ---
def tri_exact(x):
    return (2/np.pi) * np.arcsin(np.sin(0.5*x))

# --- domain ---
t = np.linspace(0, 2*np.pi, 2000)

Nmax = 6
colors = plt.cm.tab10(np.linspace(0,1,Nmax))

plt.figure(figsize=(8,5))
plt.grid(True)
plt.box(True)

for N in range(1, Nmax + 1):

    # compute coefficients up to N
    a0, an, bn = fourier_coeffs_2pi(tri_exact, N)

    # reconstruct partial sum
    fN = reconstruct_from_coeffs(t, a0, an, bn)

    plt.plot(t, fN, color=colors[N-1], linewidth=2,
             label=f"N = {N} terms")

plt.ylim([-1.2, 1.2])
plt.xlim([t[0], t[-1]])
plt.gca().tick_params(labelsize=20)
plt.legend()
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Fourier Series Partial Sums for Triangle Wave")

plt.show()

---
In the cell below, we created a sinusoidal function:

$$f(x) = A_1 \sin\bigg(\frac{2\pi}{L}n_1 x\bigg) + A_2 \sin \bigg( \frac{2\pi}{L}n_2 x \bigg),$$ 

which contains a low frequency and a high frequency components.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# define domain and discretization
# -----------------------------
L = 2.0*np.pi
gpt = 256

# x in [0, L), 256 points
x = np.linspace(0.0, L, gpt, endpoint=False)

# -----------------------------
# sinusoidal function
# -----------------------------
A1 = 5.0
A2 = 1.0
n1 = 2.0
n2 = 20.0

# --- define the function you want to expand ---
def fsin(x,L,A1,n1):
    return A1 * np.sin(n1 * x)

f1 = fsin(x,L,A1,n1)
f2 = fsin(x,L,A2,n2)

f = f1 + f2

# -----------------------------
# visualize the wave function
# -----------------------------
plt.plot(x, f, '-', linewidth=2)
plt.plot(x, f1, 'r:', linewidth=2)
plt.plot(x, f2, 'g:', linewidth=2)
plt.title('sinusoidal functions')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(True)
plt.show(block=False)

In [ ]:
import numpy as np

def f2sin(x):
    return A1*np.sin(n1 * x) + A2*np.sin(n2 * x)

len(n)
x = np.linspace(0.0, L, gpt, endpoint=False)
g = f2sin(x)
plt.plot(x,g)

N = 21
# # compute coefficients up to N
a0, an, bn = fourier_coeffs_2pi( f2sin, N)
print("a0 = ", a0)
print("an = ", an)
print("bn = ", bn)    



In [ ]:
# ----------------------------------
# Figure 3: bar plot of Fourier amplitudes
# ----------------------------------

# Suppose an, bn already computed for n = 1..N
N = len(an)

harmonics = np.arange(1, N+1)
amp_vals = np.sqrt(an**2 + bn**2)

plt.figure(figsize=(7,4))
plt.bar(harmonics, amp_vals, width=0.2, color='orange', edgecolor='k')
plt.xlabel("Harmonic index n")
plt.ylabel("Amplitude")
plt.title("Fourier harmonic amplitudes")
plt.grid(axis='y')
plt.show()

## Use FFT instead of conventional Fourier transform

In [ ]:
PyFs = np.fft.fft(g)

print(PyFs)
plt.plot(abs(PyFs))

This graph shows the contribution of each frequency to the original signal.  As you can see there are four spikes in the graph and the graph is symmetric.  Typically we often choose to just plot half of the space to see more detail. We can do this as follows:

In [ ]:
plt.plot(abs(PyFs[:int(len(PyFs)/2)]))